In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models

In [6]:
df = pd.read_csv("biofile0.csv")
df.head()

,id,lastname,usename,fullname,birthdate,birthcity,birthstate,birthcountry,deathdate,deathcity,...,last_c,debut_m,last_m,debut_u,last_u,bats,throws,height,weight,HOF
0,aardd001,Aardsma,David,David Allan Aardsma,19811227.0,Denver,Colorado,USA,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,R,R,77.0,200.0,NaN
1,aaroh101,Aaron,Hank,Henry Louis Aaron,19340205.0,Mobile,Alabama,USA,20210122.0,Atlanta,...,NaN,NaN,NaN,NaN,NaN,R,R,72.0,180.0,HOF
2,aarot101,Aaron,Tommie,Tommie Lee Aaron,19390805.0,Mobile,Alabama,USA,19840816.0,Atlanta,...,19840813.0,NaN,NaN,NaN,NaN,R,R,75.0,190.0,NaN
3,aased001,Aase,Don,Donald William Aase,19540908.0,Orange,California,USA,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,R,R,75.0,190.0,NaN
4,abada001,Abad,Andy,Fausto Andres Abad,19720825.0,Palm Beach,Florida,USA,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,L,L,73.0,184.0,NaN


In [7]:
# =========================
# 3. TARGET HANDLING
# =========================
# "HOF" -> 1, missing -> 0
df['HOF'] = df['HOF'].apply(lambda x: 1 if x == 'HOF' else 0)

# =========================
# 4. DROP HIGH-CARDINALITY COLUMNS
# =========================
drop_cols = ['id', 'lastname', 'usename', 'fullname']
df = df.drop(columns=drop_cols)

# =========================
# 5. SPLIT FEATURES / TARGET
# =========================
X = df.drop(columns=['HOF'])
y = df['HOF']

# =========================
# 6. LABEL ENCODING (MEMORY SAFE)
# =========================
for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = X[col].astype(str)
        X[col] = le.fit_transform(X[col])

# =========================
# 7. HANDLE MISSING VALUES (FEATURES ONLY)
# =========================
X = X.fillna(X.mean())

# =========================
# 8. TRAIN-TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 9. NORMALIZATION
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================
# 10. CLASS WEIGHTS
# =========================
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = {0: weights[0], 1: weights[1]}

# =========================
# 11. AUTOENCODER (SMALL)
# =========================
input_dim = X_train.shape[1]
encoding_dim = 16  # reduced size

input_layer = layers.Input(shape=(input_dim,))

encoded = layers.Dense(64, activation='relu')(input_layer)
encoded = layers.Dense(encoding_dim, activation='relu')(encoded)

decoded = layers.Dense(64, activation='relu')(encoded)
decoded = layers.Dense(input_dim, activation='linear')(decoded)

autoencoder = models.Model(input_layer, decoded)
encoder = models.Model(input_layer, encoded)

autoencoder.compile(optimizer='adam', loss='mse')

print("\nTraining Autoencoder...")
autoencoder.fit(
    X_train, X_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# =========================
# 12. RECONSTRUCTION ERROR
# =========================
reconstructed = autoencoder.predict(X_test)
reconstruction_error = np.mean((X_test - reconstructed) ** 2)

print("\nReconstruction MSE:", reconstruction_error)

# =========================
# 13. EXTRACT FEATURES
# =========================
X_train_encoded = encoder.predict(X_train)
X_test_encoded = encoder.predict(X_test)

# =========================
# 14. ANN (ENCODED FEATURES)
# =========================
model_encoded = models.Sequential([
    layers.Dense(32, activation='relu', input_shape=(encoding_dim,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model_encoded.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nTraining ANN (Encoded Features)...")
model_encoded.fit(
    X_train_encoded, y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights,
    verbose=1
)

# =========================
# 15. ANN (ORIGINAL FEATURES)
# =========================
model_original = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(input_dim,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model_original.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nTraining ANN (Original Features)...")
model_original.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights,
    verbose=1
)

# =========================
# 16. EVALUATION FUNCTION
# =========================
def evaluate(model, X, y, name):
    y_pred = (model.predict(X) > 0.5).astype(int)

    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)

    print(f"\n{name}")
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1 Score :", f1)

    return acc, prec, rec, f1

# =========================
# 17. FINAL RESULTS
# =========================
print("\n===== FINAL RESULTS =====")

evaluate(model_encoded, X_test_encoded, y_test, "Autoencoder Features")
evaluate(model_original, X_test, y_test, "Original Features")


Training Autoencoder...
Epoch 1/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.3346 - val_loss: 0.1568
Epoch 2/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1040 - val_loss: 0.0825
Epoch 3/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0633 - val_loss: 0.0572
Epoch 4/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0491 - val_loss: 0.0481
Epoch 5/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0393 - val_loss: 0.0390
Epoch 6/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0330 - val_loss: 0.0339
Epoch 7/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0290 - val_loss: 0.0326
Epoch 8/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0294 - val_loss: 0.0255
Epoch 9/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0242 - val_loss: 0.0255
Epoch 10/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0240 - val_loss: 0.0230
Epoch 11/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0220 - val_loss: 0.0196
Epoch 12/15
540/540 ━━━━━━━

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


540/540 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7059 - loss: 0.6629 - val_accuracy: 0.6280 - val_loss: 0.7397
Epoch 2/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7915 - loss: 0.5459 - val_accuracy: 0.7833 - val_loss: 0.5604
Epoch 3/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7974 - loss: 0.5327 - val_accuracy: 0.7068 - val_loss: 0.6861
Epoch 4/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8054 - loss: 0.5318 - val_accuracy: 0.8567 - val_loss: 0.5284
Epoch 5/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8164 - loss: 0.5191 - val_accuracy: 0.8449 - val_loss: 0.5115
Epoch 6/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8215 - loss: 0.5181 - val_accuracy: 0.8943 - val_loss: 0.3778
Epoch 7/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8384 - loss: 0.5024 - val_accuracy: 0.7276 - val_loss: 0.6596
Epoch 8/15
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8257 - loss: 0.5012 - val_accuracy: 0.7318 - val_

(0.9402929723715928, 0.15135135135135136, 0.875, 0.25806451612903225)